# Train a JiT-RCDM generator on Colab

One notebook for every generator in the study. Pick the frozen encoder and the
loss space in section 0; everything else is the shared recipe, so a new run is
comparable with the four reported generators at equal compute.

| Setting | Value | Note |
|---|---|---|
| denoiser | `JiT_S_16`: hidden 384, depth 12, 6 heads, patch 16, `cond_dim` 128 | same for all runs |
| batch / grad_accum | 128 / 1 | pinned, needs an A100-40GB |
| optimiser | AdamW β = (0.9, 0.95), lr 1e-4, weight decay 0 | |
| warmup / steps | 2000 / 100 000 | ≈ 10 A100-hours |
| `cfg_dropout` / EMA | 0.1 / 0.9999 | |
| t-sampler | logit-normal(−0.8, 0.8) | |
| **loss space** | **`x` (all reported generators) or `v` (upstream JiT)** | `[fix-6]`, see `prediction_mechanism_pr26.md` |

`h_dim` is never set by hand: it comes from the encoder registry via `--encoder`,
and `scripts/train.py` refuses a cache built by a different encoder.

**Before you start**, build the packed training cache locally (last cell) and
upload it to Drive. The encoder never runs on Colab.

## 0 — Configure the run

In [1]:
# ── edit these ──────────────────────────────────────────────────────────────
ENCODER     = "dinov3"   # dinov3 | dinov3_vitl16 | retfound_cfp | retfound_dinov3
LOSS_SPACE  = "v"        # "x" = objective of every reported generator; "v" = JiT's
T_MU        = -0.8       # logit-normal t-sampler; keep the defaults unless this
T_SIGMA     = 0.8        # run *is* a t-sampler ablation
TOTAL_STEPS = 100_000
USE_WANDB   = True       # False = train unlogged (section 4 is then skipped)

DRIVE_DIR   = "/content/drive/MyDrive/jit_rcdm"
REPO_SLUG   = "SeverinLe/PracticalWork_SL_JiT_RCDM"
BRANCH      = "main"
# ────────────────────────────────────────────────────────────────────────────

RUN_ID   = f"{ENCODER}_{LOSS_SPACE}loss"            # names everything below
RUN_NAME = f"S16-{ENCODER}-{LOSS_SPACE}loss-{TOTAL_STEPS // 1000}k-A100"
PACKED   = f"{DRIVE_DIR}/train_packed_{ENCODER}.pt"
CKPT_DIR = f"{DRIVE_DIR}/checkpoints_{RUN_ID}"

assert LOSS_SPACE in ("x", "v"), f"LOSS_SPACE must be 'x' or 'v', got {LOSS_SPACE!r}"
print(f"run       : {RUN_NAME}")
print(f"cache     : {PACKED}")
print(f"ckpt dir  : {CKPT_DIR}")

run       : S16-dinov3-vloss-100k-A100
cache     : /content/drive/MyDrive/jit_rcdm/train_packed_dinov3.pt
ckpt dir  : /content/drive/MyDrive/jit_rcdm/checkpoints_dinov3_vloss


## 1 — GPU check

Runtime → Change runtime type → **A100 GPU**. The batch is pinned to 128 instead
of scaled to VRAM: a smaller batch would change the optimisation budget and break
the equal-compute comparison with the reported runs.

In [2]:
import torch

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> A100.")
name = torch.cuda.get_device_name(0)
mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"PyTorch {torch.__version__} | {name} ({mem:.0f} GB)")

BATCH_SIZE = 128
if mem < 38:
    raise SystemExit(f"{mem:.0f} GB is too small for batch 128 at 224 px. Use an A100-40GB.")
print(f"batch_size = {BATCH_SIZE} (pinned)")

PyTorch 2.11.0+cu130 | NVIDIA A100-SXM4-40GB (42 GB)
batch_size = 128 (pinned)


## 2 — Mount Drive

Expected on Drive:

| File | Size | Built by |
|---|---|---|
| `jit_rcdm/train_packed_<ENCODER>.pt` | ~150 MB | `data/scripts/pack_dataset.py` (last cell) |

Checkpoints are written to `jit_rcdm/checkpoints_<ENCODER>_<LOSS>loss/`, one
folder per run, so two runs can never resume from each other's files.

In [4]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(CKPT_DIR, exist_ok=True)
assert os.path.exists(PACKED), f"Missing {PACKED}. Build it locally and upload it (last cell)."
print(f"packed cache : {os.path.getsize(PACKED) / 1e6:.0f} MB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
packed cache : 148 MB


## 3 — Clone the repository

The repository is public, so no token is needed. The cell stops if the checked-out
code predates the `--loss_space` flag. Without that check a `v` run would fail at
argument parsing, or, on an older trainer that silently ignored it, train with
x-loss under a `vloss` name.

In [5]:
import os, subprocess, sys

REPO_DIR = "/content/jit_rcdm"
URL = f"https://github.com/{REPO_SLUG}.git"

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(f"git {args[0]} failed:\n{r.stdout}{r.stderr}")
    return r.stdout.strip()

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    git("-C", REPO_DIR, "fetch", "-q", "origin", BRANCH)
    git("-C", REPO_DIR, "checkout", "-q", BRANCH)
    git("-C", REPO_DIR, "reset", "-q", "--hard", f"origin/{BRANCH}")
else:
    git("clone", "-q", "--branch", BRANCH, "--single-branch", URL, REPO_DIR)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
COMMIT = git("-C", REPO_DIR, "rev-parse", "--short", "HEAD")
print(f"{REPO_SLUG}@{BRANCH}  {git('-C', REPO_DIR, 'log', '-1', '--oneline')}")

assert "--loss_space" in open("scripts/train.py").read(), (
    "scripts/train.py on this branch has no --loss_space flag. Push the [fix-6] "
    "change (rcdm/jit.py, scripts/train.py) to the repository first."
)

from rcdm.encoders import ENCODER_NAMES, get_h_dim
assert ENCODER in ENCODER_NAMES, f"{ENCODER!r} is not in the registry: {ENCODER_NAMES}"
H_DIM = get_h_dim(ENCODER)
print(f"encoder {ENCODER!r}: h_dim = {H_DIM}")

SeverinLe/PracticalWork_SL_JiT_RCDM@main  4d33e85 Delete notebooks/colab_train_dinov3_vitl16.ipynb


AssertionError: scripts/train.py on this branch has no --loss_space flag. Push the [fix-6] change (rcdm/jit.py, scripts/train.py) to the repository first.

In [ ]:
!pip install -q transformers safetensors timm wandb
print("dependencies installed")

## 4 — Check the cache before spending 10 GPU-hours

Each of these would give a plausible-looking but meaningless run:

- **width**: `reps` must be `h_dim`-wide for this encoder;
- **alignment**: `images[i]` must belong to `reps[i]` (972 training images);
- **provenance**: the cache must come from `ENCODER`. Three encoders are
  1024-dim, so width alone cannot tell them apart;
- **not a duplicate**: identical `reps` to another encoder's cache on Drive means
  the wrong file was uploaded.

In [ ]:
import glob, os, torch

packed = torch.load(PACKED, map_location="cpu", weights_only=False)
assert "images" in packed and "reps" in packed, f"not a packed cache: {list(packed)}"
imgs, reps = packed["images"], packed["reps"]
enc = packed.get("encoder")
print(f"images  : {tuple(imgs.shape)} {imgs.dtype}")
print(f"reps    : {tuple(reps.shape)} {reps.dtype}")
print(f"encoder : {enc or '<unrecorded>'}")

assert reps.shape[1] == H_DIM, f"reps are {reps.shape[1]}-dim; {ENCODER} gives {H_DIM}"
assert imgs.shape[0] == reps.shape[0] == 972, f"expected 972 aligned pairs, got {imgs.shape[0]}/{reps.shape[0]}"
assert torch.isfinite(reps).all(), "non-finite values in reps"

same_width = [n for n in ENCODER_NAMES if get_h_dim(n) == H_DIM]
if enc is None:
    if len(same_width) > 1:
        raise SystemExit(
            f"The cache records no encoder and {H_DIM}-dim is shared by {same_width}. "
            "Re-pack it with data/scripts/pack_dataset.py so its provenance is recorded."
        )
    print(f"WARNING: no 'encoder' key; accepted only because {H_DIM}-dim is unique to {ENCODER}.")
else:
    assert enc == ENCODER, f"cache was built with {enc!r}, not {ENCODER!r}"
    print("provenance confirmed")

for other in sorted(glob.glob(f"{DRIVE_DIR}/train_packed_*.pt")):
    if os.path.samefile(other, PACKED):
        continue
    o = torch.load(other, map_location="cpu", weights_only=False)["reps"]
    if o.shape == reps.shape:
        assert not torch.allclose(o, reps, atol=1e-4), f"identical to {os.path.basename(other)}"
        print(f"differs from {os.path.basename(other)}")
del packed, imgs

## 5 — Weights & Biases

Add your key as a Colab secret (🔑 in the sidebar) named `WANDB_API_KEY`, with
notebook access ON. **Never paste a key into a cell.** Notebooks get committed,
and a key in git history stays public even after the line is deleted.

In [ ]:
WANDB_ARGS = "--no_wandb"
if USE_WANDB:
    import wandb
    from google.colab import userdata
    try:
        key = userdata.get("WANDB_API_KEY")
    except Exception as exc:
        raise SystemExit("No Colab secret WANDB_API_KEY. Add it, or set USE_WANDB = False.") from exc
    wandb.login(key=key, relogin=True)
    del key
    WANDB_ARGS = f"--wandb_project jit-rcdm --wandb_run_name {RUN_NAME}"
    print(f"W&B: https://wandb.ai/{wandb.api.default_entity}/jit-rcdm  run {RUN_NAME}")
else:
    print("W&B disabled")

## 6 — Train

**Restartable.** A checkpoint lands on Drive every 5000 steps. After a disconnect,
run sections 0–5 again and then this cell: it resumes from the newest checkpoint
in this run's folder. `train.py` refuses to resume with a `--loss_space` that
contradicts the checkpoint.

In [ ]:
import glob, os, re

ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "*step*.pt")),
               key=lambda p: int(re.search(r"step0*(\d+)", os.path.basename(p)).group(1)))
RESUME = f"--resume {ckpts[-1]}" if ckpts else ""
print(f"resuming from {os.path.basename(ckpts[-1])}" if ckpts else "starting from step 0")

!python scripts/train.py \
    --encoder        {ENCODER} \
    --reps_file      {PACKED} \
    --save_dir       {CKPT_DIR} \
    --model          S16 \
    --image_size     224 \
    --batch_size     {BATCH_SIZE} \
    --grad_accum     1 \
    --lr             1e-4 \
    --warmup_steps   2000 \
    --total_steps    {TOTAL_STEPS} \
    --save_interval  5000 \
    --log_interval   100 \
    --cfg_dropout    0.1 \
    --ema_decay      0.9999 \
    --t_mu           {T_MU} \
    --t_sigma        {T_SIGMA} \
    --loss_space     {LOSS_SPACE} \
    {WANDB_ARGS} \
    --device         cuda {RESUME}

## 7 — Write the checkpoint sidecar

Each `*.pt` under `models/` has a committed `*.cfg.json` with its architecture,
step, W&B run id and sha256, which is how a checkpoint stays identifiable once the
weights are untracked. The assertions catch a run that trained something other
than what section 0 asked for.

In [ ]:
import glob, hashlib, json, os, torch

FINAL = os.path.join(CKPT_DIR, "jit_rcdm_final.pt")
if not os.path.exists(FINAL):
    cands = sorted(glob.glob(os.path.join(CKPT_DIR, "*step*.pt")))
    assert cands, f"no checkpoint in {CKPT_DIR}"
    FINAL = cands[-1]
    print("WARNING: no final checkpoint yet; describing the newest intermediate one.")

state = torch.load(FINAL, map_location="cpu", weights_only=False)
mcfg = state["model_cfg"]
expected = {"encoder": ENCODER, "h_dim": H_DIM, "hidden_dim": 384, "cond_dim": 128,
            "t_mu": T_MU, "t_sigma": T_SIGMA, "loss_space": LOSS_SPACE}
for k, v in expected.items():
    assert mcfg.get(k) == v, f"model_cfg[{k!r}] = {mcfg.get(k)!r}, expected {v!r}"

digest = hashlib.sha256()
with open(FINAL, "rb") as fh:
    for chunk in iter(lambda: fh.read(1 << 20), b""):
        digest.update(chunk)

run_id = None
if USE_WANDB:
    import wandb
    runs = [r for r in wandb.Api().runs(f"{wandb.api.default_entity}/jit-rcdm") if r.name == RUN_NAME]
    run_id = max(runs, key=lambda r: r.created_at).id if runs else None

sidecar = {
    "file": os.path.basename(FINAL),
    "step": state.get("step"),
    "has_ema": bool(state.get("ema")),
    "wandb_run_id": run_id or state.get("wandb_run_id"),
    "git_commit": COMMIT,
    "sha256": digest.hexdigest(),
    "size_bytes": os.path.getsize(FINAL),
    "model_cfg": mcfg,
}
out = FINAL.replace(".pt", ".cfg.json")
with open(out, "w") as fh:
    json.dump(sidecar, fh, indent=2)
print(json.dumps(sidecar, indent=2))
print(f"\nwrote {out}")
print(f"Copy both into the repo as models/jit_{RUN_ID}/: the .pt stays untracked, the .cfg.json is committed.")

## Afterwards: evaluate locally

Copy `jit_rcdm_final.pt` and its `.cfg.json` into `models/jit_<ENCODER>_<LOSS>loss/`
and run the probes with the same settings as the reported runs, so the only
difference is the generator:

```bash
CKPT=models/jit_dinov3_vloss/jit_rcdm_final.pt
python experiments/E2_sample_variability.py --checkpoint $CKPT --encoder dinov3 \
    --n_per_class 6 --n_samples 8 --seed 0 --tag vloss_seed0 --device mps
python experiments/E1_fidelity.py --checkpoint $CKPT --encoder dinov3 \
    --n_per_class 40 --seed 0 --tag vloss_seed0 --device mps
```

For the loss-space ablation, compare E2's within-`h` variability with the x-loss
generator for the same encoder. E1 checks that fidelity did not drop.

---

### Building the packed cache locally (before this notebook)

```bash
# 1. representations (skip if data/processed/messidor2/<ENCODER>/train_reps.pt exists)
python data/scripts/precompute_reps.py --encoder dinov3 \
    --data_dir data/raw/messidor2/train \
    --out_file data/processed/messidor2/dinov3/train_reps.pt --device mps

# 2. pack images + reps into one file; this also records the encoder name
python data/scripts/pack_dataset.py \
    --reps_file data/processed/messidor2/dinov3/train_reps.pt \
    --out_file  data/processed/messidor2/dinov3/train_packed.pt \
    --image_size 224
```

Upload `train_packed.pt` to `MyDrive/jit_rcdm/` as `train_packed_<ENCODER>.pt`.
Re-pack an older file if section 4 reports `<unrecorded>`: packed caches built
before the `encoder` key existed cannot prove which encoder made them.